# Advanced Dividend Reinvestment Calculator

Enter a ticker and a few inputs in the cell below; the share price and dividend yield are pulled from Yahoo Finance. The yield is annualized by taking the most recent dividend payment and multiplying it by `dividend_frequency`.

**Assumptions**
- The annual dividend yield and annual price growth are stated as annual figures and are split evenly across the chosen compounding periods.
- Dividends are reinvested at the prevailing share price at the end of each period.
- No taxes or fees.

In [ ]:
# --- Key inputs (edit these) ---
# Run once: pip install yfinance
import yfinance as yf

ticker = 'NEHI'                 # ticker symbol
initial_investment = 200000.0   # dollars invested up front
annual_price_growth = 0.0      # 6% annual share price growth
years = 8                      # holding period
PAYOUTS_PER_YEAR = {'daily': 365, 'monthly': 12, 'quarterly': 4, 'yearly': 1}
compounding_period = 'monthly'
dividend_frequency = PAYOUTS_PER_YEAR[compounding_period]

# --- Pull share price and dividend yield from Yahoo Finance --------------------
tk = yf.Ticker(ticker)
info = tk.info
initial_share_price = info.get('regularMarketPrice') or info.get('previousClose')

dividends = tk.dividends
if dividends.empty:
    raise ValueError(f"No dividend history found for {ticker}")
last_dividend = float(dividends.iloc[-1])
last_dividend_date = dividends.index[-1].date()
annualized_dividend = last_dividend * dividend_frequency
annual_dividend_yield = annualized_dividend / initial_share_price

print(f"Ticker:           {ticker} ({info.get('shortName', 'n/a')})")
print(f"Share price:      ${initial_share_price:,.2f}")
print(f"Last dividend:    ${last_dividend:.4f}  (on {last_dividend_date})")
print(f"Annualized (x{dividend_frequency}): ${annualized_dividend:.4f}")
print(f"Dividend yield:   {annual_dividend_yield:.2%}")

def simulate_drip(initial_investment, initial_share_price,
                  annual_dividend_yield, annual_price_growth,
                  years, compounding_period):
    """DRIP simulation with configurable compounding period."""
    if compounding_period not in PAYOUTS_PER_YEAR:
        raise ValueError(
            f"compounding_period must be one of {list(PAYOUTS_PER_YEAR)}")

    n = PAYOUTS_PER_YEAR[compounding_period]
    total_periods = years * n
    period_growth = (1 + annual_price_growth) ** (1 / n) - 1
    period_yield = annual_dividend_yield / n

    shares = initial_investment / initial_share_price
    price = initial_share_price
    rows = [{'period': 0, 'year': 0.0, 'price': price,
             'shares': shares, 'dividend': 0.0, 'value': shares * price}]
    for p in range(1, total_periods + 1):
        price *= (1 + period_growth)
        dividend_cash = shares * price * period_yield
        shares += dividend_cash / price
        rows.append({'period': p, 'year': p / n, 'price': price,
                     'shares': shares, 'dividend': dividend_cash,
                     'value': shares * price})
    return rows, n

results, n = simulate_drip(
    initial_investment, initial_share_price,
    annual_dividend_yield, annual_price_growth,
    years, compounding_period,
)

# --- Annual snapshots (one row per full year, regardless of period) ---
print(f"\nCompounding: {compounding_period} ({n} periods/year)\n")
print(f"{'Year':>4} {'Price':>10} {'Shares':>12} {'Value':>14}")
print('-' * 44)
for r in results:
    if r['period'] % n == 0:
        print(f"{int(r['year']):>4} {r['price']:>10.2f} "
              f"{r['shares']:>12.4f} {r['value']:>14.2f}")

# --- Summary ------------------------------------------------------------
final = results[-1]
total_dividends = sum(r['dividend'] for r in results)
total_return = (final['value'] / initial_investment) - 1
cagr = (final['value'] / initial_investment) ** (1 / years) - 1

print(f"Ticker:               {ticker}")
print(f"Compounding period:   {compounding_period}")
print(f"Initial investment:   ${initial_investment:,.2f}")
print(f"Final value:          ${final['value']:,.2f}")
print(f"Total dividends paid: ${total_dividends:,.2f}")
print(f"Final share count:    {final['shares']:,.4f}")
print(f"Total return:         {total_return:.2%}")
print(f"CAGR:                 {cagr:.2%}")

In [ ]:
# --- Visualize the returns ---
import matplotlib.pyplot as plt

# x-axis in units of the compounding period (e.g. months)
PERIOD_LABEL = {
    'daily': ('day', 'Days'),
    'monthly': ('month', 'Months'),
    'quarterly': ('quarter', 'Quarters'),
    'yearly': ('year', 'Years'),
}
unit_singular, unit_plural = PERIOD_LABEL[compounding_period]

xs = [r['period'] for r in results]
values = [r['value'] for r in results]
cum_dividends = []
running = 0.0
for r in results:
    running += r['dividend']
    cum_dividends.append(running)
# Capital appreciation = value - initial - cumulative dividends paid out
# (since dividends are reinvested, this is really price-driven growth)
price_growth_component = [v - initial_investment - d for v, d in zip(values, cum_dividends)]
# Cumulative initial+dividends boundary (top of the dividend layer in the stack)
initial_plus_dividends = [initial_investment + d for d in cum_dividends]

# Marker size shrinks as point density grows so daily compounding stays readable
marker_size = max(2, min(6, int(400 / len(xs))))

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

# Top: stacked area of principal / dividends / price growth, with line+marker overlays
ax = axes[0]
ax.stackplot(
    xs,
    [initial_investment] * len(xs),
    cum_dividends,
    price_growth_component,
    labels=['Initial investment', 'Reinvested dividends', 'Price appreciation'],
    colors=['#4C72B0', '#55A868', '#C44E52'],
    alpha=0.6,
)
ax.plot(xs, [initial_investment] * len(xs),
        color='#2A4A75', lw=1.2, marker='o', markersize=marker_size)
ax.plot(xs, initial_plus_dividends,
        color='#2F6B3E', lw=1.2, marker='o', markersize=marker_size,
        label='Initial + dividends')
ax.plot(xs, values,
        color='black', lw=1.5, marker='o', markersize=marker_size,
        label='Portfolio value')
ax.set_ylabel('Portfolio value ($)')
ax.set_title(f"{ticker} DRIP projection — ${initial_investment:,.0f} over {years} years")
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Bottom: dividend income per period (bars) plus a marker on each
ax = axes[1]
dividends_per_period = [r['dividend'] for r in results]
ax.bar(xs, dividends_per_period, width=1.0, color='#55A868', alpha=0.8)
ax.plot(xs, dividends_per_period,
        color='#2F6B3E', lw=1.0, marker='o', markersize=marker_size,
        label=f'Dividend per {unit_singular}')
ax.set_xlabel(f'{unit_plural} since start')
ax.set_ylabel(f'Dividend per {unit_singular} ($)')
ax.set_title('Dividend income per period (reinvested)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()